# EEGNet Seizure Prediction
Cross-patient benchmarking on CHB-MIT · 20 seeds · mean ± std

In [1]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
from torch.optim import AdamW
from sklearn.metrics import roc_auc_score
from data_utils import get_dataloaders_for_seed, SEEDS, DATA_DIR
from eval_utils import find_youden_threshold, full_evaluate

DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_CHANNELS   = 18
WIN          = 20 * 256
BATCH_SIZE   = 128
MAX_EPOCHS   = 100
PATIENCE     = 20
LR           = 1e-3
WEIGHT_DECAY = 1e-4
OUT_DIR      = r'D:\seizure_results\eegnet'

print(f'Device : {DEVICE}')
print(f'Seeds  : {len(SEEDS)}')
os.makedirs(OUT_DIR, exist_ok=True)

Device : cuda
Seeds  : 20


In [2]:
class EEGNet(nn.Module):
    """
    EEGNet — Lawhern et al. (2018), Journal of Neural Engineering.
    Original hyperparameters: F1=8, D=2, dropout=0.5.

    kern_length=127 (odd) instead of 128:
    PyTorch Conv2d with symmetric padding=(0, kern//2) preserves the time
    dimension exactly only for odd kernels. Using 127 vs 128 is functionally
    equivalent (both span ~0.5 s at 256 Hz) and avoids the T→T+1 artefact
    that occurs with even kernels under symmetric padding.

    Input  : (B, 18, 5120)
    Output : (B, 2)
    """
    def __init__(self,
                 n_channels   = N_CHANNELS,
                 n_times      = WIN,
                 F1           = 8,
                 D            = 2,
                 kern_length  = 127,   # odd → symmetric padding keeps T exact
                 dropout      = 0.5,
                 n_classes    = 2):
        super().__init__()
        F2 = F1 * D

        # Block 1: temporal + depthwise spatial convolution
        self.block1 = nn.Sequential(
            nn.Conv2d(1, F1, kernel_size=(1, kern_length),
                      padding=(0, kern_length // 2), bias=False),
            nn.BatchNorm2d(F1),
            nn.Conv2d(F1, F1 * D, kernel_size=(n_channels, 1),
                      groups=F1, bias=False),
            nn.BatchNorm2d(F1 * D),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 4)),
            nn.Dropout(dropout),
        )

        # Block 2: separable convolution
        # kernel=(1,15) is odd — no T+1 issue
        self.block2 = nn.Sequential(
            nn.Conv2d(F2, F2, kernel_size=(1, 15),
                      padding=(0, 7), groups=F2, bias=False),
            nn.Conv2d(F2, F2, kernel_size=(1, 1), bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 8)),
            nn.Dropout(dropout),
        )

        # Compute flattened size dynamically
        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_channels, n_times)
            flat  = self.block2(self.block1(dummy)).view(1, -1).shape[1]

        self.classifier = nn.Linear(flat, n_classes)

    def forward(self, x):
        x = x.unsqueeze(1)           # (B, 1, C, T)
        x = self.block1(x)
        x = self.block2(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)


dummy = torch.zeros(4, N_CHANNELS, WIN)
model_test = EEGNet()
print('Output shape:', model_test(dummy).shape)   # expect (4, 2)
n_params = sum(p.numel() for p in model_test.parameters() if p.requires_grad)
print(f'Trainable parameters: {n_params:,}')

Output shape: torch.Size([4, 2])
Trainable parameters: 7,002


In [3]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * len(y)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def collect_probs(model, loader):
    model.eval()
    all_probs, all_labels = [], []
    for x, y in loader:
        logits = model(x.to(DEVICE))
        probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        all_probs.append(probs)
        all_labels.append(y.numpy())
    return np.concatenate(all_probs), np.concatenate(all_labels)
print('Helpers defined.')

Helpers defined.


In [4]:
def run_seed(seed):
    print(f"\n{'='*60}")
    print(f"  Seed {seed}")
    print(f"{'='*60}")

    (train_loader, val_loader, test_loader,
     n_train, n_val, n_test,
     n_pre, n_inter) = get_dataloaders_for_seed(seed, DATA_DIR, BATCH_SIZE)

    _bx, _by = next(iter(train_loader))
    _n1 = int(_by.sum())
    print(f"  Batch check : {_n1}/{len(_by)} pre-ictal ({100*_n1/len(_by):.0f}%)")
    del _bx, _by

    model     = EEGNet().to(DEVICE)
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=5, min_lr=1e-7)
    criterion = nn.CrossEntropyLoss()

    best_val_auc  = 0.0
    best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    patience_left = PATIENCE

    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
        val_probs, val_labels = collect_probs(model, val_loader)
        val_auc = roc_auc_score(val_labels, val_probs)
        scheduler.step(val_auc)
        current_lr = optimizer.param_groups[0]['lr']

        print(f"  Epoch {epoch:3d} | loss {train_loss:.4f} | "
              f"val AUC {val_auc:.4f} | lr {current_lr:.2e}")

        if val_auc > best_val_auc:
            best_val_auc  = val_auc
            best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_left = PATIENCE
        else:
            patience_left -= 1
            if patience_left == 0:
                print(f"  Early stopping at epoch {epoch}.")
                break

    model.load_state_dict(best_state)
    model.to(DEVICE)

    val_probs, val_labels = collect_probs(model, val_loader)
    threshold = find_youden_threshold(val_labels, val_probs)
    print(f"  Youden threshold (val): {threshold:.4f}")

    test_probs, test_labels = collect_probs(model, test_loader)
    m = full_evaluate(test_labels, test_probs, threshold, stride_s=300)

    print(f"\n  [Seed {seed}] TEST  AUC {m['auc']:.4f} | "
          f"Sen {m['sensitivity']:.4f} | Spe {m['specificity']:.4f} | "
          f"Prec {m['precision']:.4f} | F1 {m['f1']:.4f} | "
          f"FAR {m['far']:.3f}/h | "
          f"EvtSen {m['event_sensitivity']:.3f} ({m['n_events']} events)")

    torch.save(best_state, os.path.join(OUT_DIR, f'seed{seed}_best.pt'))

    return {
        'seed':               seed,
        'test_auc':           m['auc'],
        'test_sen':           m['sensitivity'],
        'test_spe':           m['specificity'],
        'test_precision':     m['precision'],
        'test_f1':            m['f1'],
        'far':                m['far'],
        'event_sensitivity':  m['event_sensitivity'],
        'n_events':           m['n_events'],
        'threshold':          m['threshold'],
        'best_val_auc':       best_val_auc,
    }

print('run_seed defined.')

run_seed defined.


In [5]:
all_results = []
for s in SEEDS:
    result = run_seed(s)
    all_results.append(result)

results_path = os.path.join(OUT_DIR, 'results.json')
with open(results_path, 'w') as _f:
    json.dump(
        [{k: (int(v) if k in ('seed', 'n_events') else float(v))
          for k, v in r.items()}
         for r in all_results],
        _f, indent=2,
    )
print(f'\nAll results saved to {results_path}')


  Seed 42

[seed=42]  train=15pts  val=4pts  test=4pts
  train  total=11726  pre=5074  inter=6652  ratio=1:1
  val    total=3521  pre=2084  inter=1437
  test   total=2177  pre=1334  inter=843
  Batch check : 61/128 pre-ictal (48%)
  Epoch   1 | loss 0.6937 | val AUC 0.5473 | lr 1.00e-03
  Epoch   2 | loss 0.6193 | val AUC 0.5385 | lr 1.00e-03
  Epoch   3 | loss 0.5620 | val AUC 0.5220 | lr 1.00e-03
  Epoch   4 | loss 0.5396 | val AUC 0.5115 | lr 1.00e-03
  Epoch   5 | loss 0.5227 | val AUC 0.5067 | lr 1.00e-03
  Epoch   6 | loss 0.5088 | val AUC 0.4957 | lr 1.00e-03
  Epoch   7 | loss 0.5007 | val AUC 0.5030 | lr 5.00e-04
  Epoch   8 | loss 0.4828 | val AUC 0.5030 | lr 5.00e-04
  Epoch   9 | loss 0.4742 | val AUC 0.5078 | lr 5.00e-04
  Epoch  10 | loss 0.4729 | val AUC 0.5107 | lr 5.00e-04
  Epoch  11 | loss 0.4631 | val AUC 0.5108 | lr 5.00e-04
  Epoch  12 | loss 0.4643 | val AUC 0.5178 | lr 5.00e-04
  Epoch  13 | loss 0.4540 | val AUC 0.5262 | lr 2.50e-04
  Epoch  14 | loss 0.4508 |

In [6]:
metrics = ['test_auc', 'test_sen', 'test_spe', 'test_precision',
           'test_f1', 'far', 'event_sensitivity']
labels  = ['AUC', 'Sensitivity (win)', 'Specificity', 'Precision',
           'F1', 'FAR (/h)', 'Sensitivity (event)']

print(f'EEGNet  Cross-Patient Results (mean ± std, n={len(all_results)} seeds)')
print('-' * 55)
for m, l in zip(metrics, labels):
    vals = np.array([r[m] for r in all_results])
    print(f'  {l:<22s}: {vals.mean():.4f} ± {vals.std(ddof=1):.4f}')

EEGNet  Cross-Patient Results (mean ± std, n=20 seeds)
-------------------------------------------------------
  AUC                   : 0.5642 ± 0.0718
  Sensitivity (win)     : 0.4691 ± 0.2532
  Specificity           : 0.6043 ± 0.2255
  Precision             : 0.5763 ± 0.1056
  F1                    : 0.4738 ± 0.1669
  FAR (/h)              : 4.7483 ± 2.7058
  Sensitivity (event)   : 0.9229 ± 0.1419
